In [0]:
applicants_df = spark.table("all_applicants")
applicants_df.head(5)

# applicants_df.printSchema()

In [0]:
from pyspark.sql.functions import *

candidate_df = (applicants_df.withColumn("email",lower(trim(col("email")))).dropDuplicates()) #

candidate_df.show()

In [0]:
academy_df = spark.table("all_academy")
academy_df.show()



+-----------------+------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+-----------+--------------+-------------+---------------+-----------+--------------+--------------------+-----------+--------------+-------------+---------------+-----------+--------------+------------+---------------+--------------+----------------+------------+---------------+
|             name|     trainer|Analytic_W1|Independent_W1|Determined_W1|Professional_W1|Studious_W1

In [0]:
competency_columns = [c for c in academy_df.columns if "_W" in c]
competencies = sorted(set(c.split("_")[0]for c in competency_columns))
competency_df = spark.createDataFrame([(c,) for c in competency_columns],["competency_name"])
print(competencies)


['Analytic', 'Determined', 'Imaginative', 'Independent', 'Professional', 'Studious']


In [0]:
competency_df = spark.createDataFrame([(c,) for c in competencies],["competency_name"])
competency_df.show()

+---------------+
|competency_name|
+---------------+
|       Analytic|
|     Determined|
|    Imaginative|
|    Independent|
|   Professional|
|       Studious|
+---------------+



In [0]:
weeks = sorted(set(int(c.split("_W")[1])for c in competency_columns))
print(weeks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [0]:

week_df = spark.createDataFrame([(w,) for w in weeks],["week"])
week_df.show()


+----+
|week|
+----+
|   1|
|   2|
|   3|
|   4|
|   5|
|   6|
|   7|
|   8|
|   9|
|  10|
+----+



In [0]:
trainer_df = (academy_df.select("trainer").distinct())
trainer_df.show()

+------------------+
|           trainer|
+------------------+
|      Gregor Gomez|
|        Bruce Lugo|
|     Neil Mccarthy|
|    Rachel Richard|
|      Hamzah Melia|
|     Burhan Milner|
|        Elly Kelly|
|          Ely Kely|
|     Trixie Orange|
|      John Sandbox|
|   Edward Reinhart|
|       Lucy Foster|
|   Gina Cartwright|
|      Eshal Brandt|
|   Macey Broughton|
|       Igor Coates|
|Mohammad Velazquez|
|   Martina Meadows|
+------------------+



In [0]:
from pyspark.sql.functions import row_number  
from pyspark.sql.window import Window 

trainer_df = trainer_df.withColumn("trainer_id",row_number().over(Window.orderBy("trainer")))
trainer_df = trainer_df.select("trainer_id","trainer")
trainer_df.show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+------------------+
|trainer_id|           trainer|
+----------+------------------+
|         1|        Bruce Lugo|
|         2|     Burhan Milner|
|         3|   Edward Reinhart|
|         4|        Elly Kelly|
|         5|          Ely Kely|
|         6|      Eshal Brandt|
|         7|   Gina Cartwright|
|         8|      Gregor Gomez|
|         9|      Hamzah Melia|
|        10|       Igor Coates|
|        11|      John Sandbox|
|        12|       Lucy Foster|
|        13|   Macey Broughton|
|        14|   Martina Meadows|
|        15|Mohammad Velazquez|
|        16|     Neil Mccarthy|
|        17|    Rachel Richard|
|        18|     Trixie Orange|
+----------+------------------+



In [0]:
week = [(i,)for i in range(1,11)]
week_df = spark.createDataFrame(week,["week"])
week_df.show()

+----+
|week|
+----+
|   1|
|   2|
|   3|
|   4|
|   5|
|   6|
|   7|
|   8|
|   9|
|  10|
+----+



creating a weekly review 

In [0]:
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

academy_df = spark.table("all_academy")

# Get unique trainee/trainer combinations
base_df = (academy_df.select("name", "trainer").distinct())
# Create one row per trainee per week
weekly_review_df = None

# base_df.show()
for week in weeks:
    temp_df = base_df.withColumn("week",lit(week))
    if weekly_review_df is None:
        weekly_review_df = temp_df
    else:
        weekly_review_df = weekly_review_df.union(temp_df)

weekly_review_df = weekly_review_df.withColumn("review_id",row_number().over(Window.orderBy("name", "week")))

# Reorder columns
weekly_review_df = weekly_review_df.select(
    "review_id",
    "name",
    "trainer",
    "week"
)

weekly_review_df.show()



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+--------------+-------------+----+
|review_id|          name|      trainer|week|
+---------+--------------+-------------+----+
|        1| Adah Spencers|Neil Mccarthy|   1|
|        2| Adah Spencers|Neil Mccarthy|   2|
|        3| Adah Spencers|Neil Mccarthy|   3|
|        4| Adah Spencers|Neil Mccarthy|   4|
|        5| Adah Spencers|Neil Mccarthy|   5|
|        6| Adah Spencers|Neil Mccarthy|   6|
|        7| Adah Spencers|Neil Mccarthy|   7|
|        8| Adah Spencers|Neil Mccarthy|   8|
|        9| Adah Spencers|Neil Mccarthy|   9|
|       10| Adah Spencers|Neil Mccarthy|  10|
|       11|Adolph Andreia|   Bruce Lugo|   1|
|       12|Adolph Andreia|   Bruce Lugo|   2|
|       13|Adolph Andreia|   Bruce Lugo|   3|
|       14|Adolph Andreia|   Bruce Lugo|   4|
|       15|Adolph Andreia|   Bruce Lugo|   5|
|       16|Adolph Andreia|   Bruce Lugo|   6|
|       17|Adolph Andreia|   Bruce Lugo|   7|
|       18|Adolph Andreia|   Bruce Lugo|   8|
|       19|Adolph Andreia|   Bruce

In [0]:
weekly_review_df.count()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


3970